# Class Activity: App Reviews Data Analysis

This notebook answers five questions using `app_reviews_demo.csv`.

In [3]:
import pandas as pd

# Load the dataset
df = pd.read_csv("app_reviews_demo.csv")

# Optional display settings for readability
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Rows: 500
Columns: 10


## 1) What does the dataset look like?

In [4]:
# Script 1: quick structure preview table
if "df" not in globals():
    import pandas as pd
    df = pd.read_csv("app_reviews_demo.csv")

preview_table = df.head(10)

print("Column names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)

preview_table

Column names:
['id', 'app', 'category', 'rating', 'review', 'date', 'helpful_votes', 'verified_purchase', 'device_type', 'app_version']

Data types:
id                   int64
app                    str
category               str
rating               int64
review                 str
date                   str
helpful_votes        int64
verified_purchase     bool
device_type            str
app_version            str
dtype: object


,id,app,category,rating,review,date,helpful_votes,verified_purchase,device_type,app_version
0,1,Fieldkit,field research,1,Auto-transcription accuracy on accented speake...,2023-03-31,37,True,mobile,2.5.0
1,2,Fieldkit,field research,2,Search results are slow when the repository is...,2024-07-28,12,True,mobile,2.5.3
2,3,Lookback,user research,4,One-click export to Notion is a feature I use ...,2024-03-08,21,True,desktop,5.2.0
3,4,Dovetail,research repository,5,My whole team can comment on the same session ...,2023-12-19,38,True,NaN,2.0.0
4,5,Fieldkit,field research,5,Works offline and syncs when I get back to WiF...,2024-01-23,5,True,desktop,2.5.3
5,6,Lookback,user research,5,The guest access feature is perfect for extern...,2024-11-08,14,False,NaN,5.3.1
6,7,Fieldkit,field research,5,Project folders keep multi-phase studies manag...,2023-06-16,23,True,mobile,3.0.0
7,8,Maze,usability testing,5,I was running my first study within 20 minutes...,2023-06-25,34,False,NaN,4.1.0
8,9,Dovetail,research repository,4,The guest access feature is perfect for extern...,2023-08-23,2,False,desktop,2.0.0
9,10,Miro,collaborative whiteboard,3,"Works fine — nothing remarkable, nothing broken.",2024-08-03,45,True,NaN,9.4.0


## 2) What is the distribution of the most important column?

For app review datasets, `rating` is usually the key outcome column.

In [5]:
# Script 2: distribution table for ratings
if "df" not in globals():
    import pandas as pd
    df = pd.read_csv("app_reviews_demo.csv")

rating_distribution = (
    df["rating"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("rating")
    .reset_index(name="count")
)
rating_distribution["percent"] = (rating_distribution["count"] / len(df) * 100).round(2)

rating_distribution

,rating,count,percent
0,1,29,5.8
1,2,43,8.6
2,3,61,12.2
3,4,160,32.0
4,5,207,41.4


## 3) Filter to a meaningful subset: what is in it?

Subset used: reviews with low ratings (`rating <= 2`) and high visibility (`helpful_votes >= 20`).

In [6]:
# Script 3: meaningful subset table
if "df" not in globals():
    import pandas as pd
    df = pd.read_csv("app_reviews_demo.csv")

subset = df[(df["rating"] <= 2) & (df["helpful_votes"] >= 20)].copy()

subset_summary = (
    subset[["app", "category"]]
    .value_counts()
    .reset_index(name="review_count")
    .sort_values("review_count", ascending=False)
)

print(f"Subset rows: {len(subset)}")
print("\nTop app/category combinations in this subset:")
subset_summary.head(10)

print("\nExample rows from the subset:")
subset[["id", "app", "category", "rating", "helpful_votes", "review"]].head(10)

Subset rows: 38

Top app/category combinations in this subset:

Example rows from the subset:


,id,app,category,rating,helpful_votes,review
0,1,Fieldkit,field research,1,37,Auto-transcription accuracy on accented speake...
51,52,Maze,usability testing,1,32,Session sharing links occasionally expire befo...
110,111,Miro,collaborative whiteboard,2,35,The browser extension stops recording silently...
117,118,Miro,collaborative whiteboard,1,41,"The difference between clips, highlights, and ..."
119,120,Miro,collaborative whiteboard,2,41,The tag hierarchy system takes a few sessions ...
150,151,Fieldkit,field research,2,31,"The difference between clips, highlights, and ..."
162,163,Fieldkit,field research,2,21,The browser extension stops recording silently...
164,165,Lookback,user research,1,31,No way to pay monthly — annual commitment only...
171,172,Maze,usability testing,2,33,Jumping from the free tier to paid is a signif...
178,179,Maze,usability testing,2,29,Guest seats should be free — charging for view...


## 4) Group by a category and find the average of a numeric column

In [7]:
# Script 4: average rating and helpful votes by category
if "df" not in globals():
    import pandas as pd
    df = pd.read_csv("app_reviews_demo.csv")

category_averages = (
    df.groupby("category", dropna=False)
      .agg(
          avg_rating=("rating", "mean"),
          avg_helpful_votes=("helpful_votes", "mean"),
          review_count=("id", "count")
      )
      .reset_index()
      .sort_values("avg_rating", ascending=False)
)

category_averages[["avg_rating", "avg_helpful_votes"]] = (
    category_averages[["avg_rating", "avg_helpful_votes"]].round(2)
)

category_averages

,category,avg_rating,avg_helpful_votes,review_count
2,research repository,4.12,24.60,89
0,collaborative whiteboard,4.02,25.08,121
3,usability testing,4.00,21.97,93
4,user research,3.90,21.88,105
1,field research,3.67,23.57,92


## 5) Where are the missing values? Are any columns incomplete?

In [8]:
# Script 5: missing values audit table
if "df" not in globals():
    import pandas as pd
    df = pd.read_csv("app_reviews_demo.csv")

missing_table = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

incomplete_columns = missing_table[missing_table["missing_count"] > 0]

print("Missing values by column:")
missing_table

print("\nColumns with missing values:")
incomplete_columns

# Optional: view a few rows that contain any missing value
rows_with_missing = df[df.isna().any(axis=1)]
print(f"\nRows containing at least one missing value: {len(rows_with_missing)}")
rows_with_missing.head(10)

Missing values by column:

Columns with missing values:

Rows containing at least one missing value: 165


,id,app,category,rating,review,date,helpful_votes,verified_purchase,device_type,app_version
3,4,Dovetail,research repository,5,My whole team can comment on the same session ...,2023-12-19,38,True,NaN,2.0.0
5,6,Lookback,user research,5,The guest access feature is perfect for extern...,2024-11-08,14,False,NaN,5.3.1
7,8,Maze,usability testing,5,I was running my first study within 20 minutes...,2023-06-25,34,False,NaN,4.1.0
9,10,Miro,collaborative whiteboard,3,"Works fine — nothing remarkable, nothing broken.",2024-08-03,45,True,NaN,9.4.0
12,13,Fieldkit,field research,3,Solid core product with rough edges in the adv...,2024-11-27,27,True,desktop,NaN
14,15,Lookback,user research,5,The timeline view across a research program is...,2024-04-09,0,False,mobile,NaN
18,19,Lookback,user research,5,Automatic session indexing makes it easy to fi...,2024-06-29,8,True,NaN,5.3.1
23,24,Fieldkit,field research,5,Session recordings sync instantly — no waiting...,2023-08-08,34,True,NaN,NaN
27,28,Maze,usability testing,4,Project folders keep multi-phase studies manag...,2024-07-17,6,True,desktop,NaN
28,29,Lookback,user research,5,Being able to rewatch sessions at 1.5x speed h...,2023-08-07,25,False,NaN,5.2.0
